# Notebook 5: GPU Performance Benchmarking

Monte Carlo simulation is embarrassingly parallel — each path is independent and can
be assigned to its own thread. This notebook measures what that means in practice:
how much faster the GPU kernel is compared to the CPU implementation, how the speedup
scales with path count, and which option types benefit most from the hardware.

In [ ]:
import sys, os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(),
    '..' if os.path.basename(os.getcwd()) == 'notebooks' else '.'))
sys.path.insert(0, os.path.join(REPO_ROOT, 'build'))

import mc_engine
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import time

S, K, r, sigma, T = 100.0, 105.0, 0.05, 0.20, 1.0
STEPS = 252

def time_ms(fn):
    """Run fn() once as warm-up, then time it and return (result, elapsed_ms)."""
    fn()                             # warm up
    t0  = time.perf_counter()
    res = fn()
    return res, (time.perf_counter() - t0) * 1000.0

# GPU warm-up: first CUDA call includes context init overhead
_ = mc_engine.GBMPricer(spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
                         n_paths=1000, n_steps=10, use_gpu=True).price_european_call()
print('GPU warm-up done.')

## Section 1 — CPU vs GPU Speedup vs Path Count

GPU parallelism is most effective when there is enough work to saturate the hardware.
For small path counts the overhead of kernel launch and memory transfer dominates;
for large counts the GPU's massively parallel execution wins decisively.

In [ ]:
path_counts = [10_000, 100_000, 1_000_000, 5_000_000, 10_000_000]

cpu_times_ms = []
gpu_times_ms = []

for n in path_counts:
    # GPU
    _, t_gpu = time_ms(lambda n=n: mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=n, n_steps=STEPS, use_gpu=True).price_european_call())
    gpu_times_ms.append(t_gpu)

    # CPU — cap at 1M to avoid long waits
    if n <= 1_000_000:
        _, t_cpu = time_ms(lambda n=n: mc_engine.GBMPricer(
            spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
            n_paths=n, n_steps=STEPS, use_gpu=False).price_european_call())
        cpu_times_ms.append(t_cpu)
    else:
        cpu_times_ms.append(float('nan'))   # too slow to time

    label = f"{n:>10,}: GPU={t_gpu:7.1f} ms"
    if not np.isnan(cpu_times_ms[-1]):
        label += f", CPU={cpu_times_ms[-1]:8.1f} ms, speedup={cpu_times_ms[-1]/t_gpu:.1f}×"
    print(label)

ns = np.array(path_counts, dtype=float)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.loglog(ns, gpu_times_ms, 'r-o', linewidth=2, markersize=6, label='GPU')
valid_cpu = [(n, t) for n, t in zip(ns, cpu_times_ms) if not np.isnan(t)]
if valid_cpu:
    ns_cpu, ts_cpu = zip(*valid_cpu)
    ax1.loglog(ns_cpu, ts_cpu, 'b-o', linewidth=2, markersize=6, label='CPU')
ax1.set_xlabel('Number of Paths', fontsize=12)
ax1.set_ylabel('Wall Time (ms)', fontsize=12)
ax1.set_title('CPU vs GPU: Wall Time', fontsize=12)
ax1.legend(fontsize=10)
ax1.grid(True, which='both', alpha=0.3)

speedups = [c / g for c, g in zip(cpu_times_ms, gpu_times_ms)
            if not np.isnan(c)]
ns_spd   = [n for n, c in zip(ns, cpu_times_ms) if not np.isnan(c)]
ax2.semilogx(ns_spd, speedups, 'g-o', linewidth=2, markersize=6)
ax2.axhline(1.0, color='black', linestyle='--', linewidth=0.8)
ax2.set_xlabel('Number of Paths', fontsize=12)
ax2.set_ylabel('GPU Speedup (×)', fontsize=12)
ax2.set_title('GPU Speedup vs Path Count', fontsize=12)
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

## Section 2 — Scaling Across Option Types

Different option types have different computational profiles:

- **European**: only needs the terminal value → minimal memory, fastest
- **Asian**: needs a running sum over all steps → more work per path
- **Barrier**: needs to check the barrier at every step → similar cost to Asian
- **Greeks**: runs 8 bumped scenarios in one batched kernel → most computation

GPU benefits path-dependent payoffs especially well since all steps are independent
across paths and can be parallelised simultaneously.

In [ ]:
N = 1_000_000

option_types = [
    ('European',  lambda gpu: mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=STEPS, use_gpu=gpu).price_european_call()),
    ('Asian',     lambda gpu: mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=STEPS, use_gpu=gpu).price_asian_call()),
    ('Barrier',   lambda gpu: mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=STEPS, use_gpu=gpu).price_barrier_call(
            barrier=115.0, barrier_type=mc_engine.BarrierType.UP_AND_OUT)),
    ('Greeks',    lambda gpu: mc_engine.GBMPricer(
        spot=S, strike=K, rate=r, volatility=sigma, maturity=T,
        n_paths=N, n_steps=STEPS, use_gpu=gpu).compute_greeks()),
]

results = []
for name, fn in option_types:
    _, t_gpu = time_ms(lambda fn=fn: fn(True))
    _, t_cpu = time_ms(lambda fn=fn: fn(False))
    results.append((name, t_cpu, t_gpu, t_cpu / t_gpu, N / (t_gpu / 1000.0)))
    print(f'  {name:<10}: CPU={t_cpu:8.1f} ms, GPU={t_gpu:6.1f} ms, speedup={t_cpu/t_gpu:.1f}×')

labels    = [r[0] for r in results]
cpu_ms    = [r[1] for r in results]
gpu_ms    = [r[2] for r in results]

x  = np.arange(len(labels))
w  = 0.35
colors_cpu = '#4C72B0'
colors_gpu = '#DD8452'

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - w/2, cpu_ms, w, label='CPU', color=colors_cpu, alpha=0.9)
b2 = ax.bar(x + w/2, gpu_ms, w, label='GPU', color=colors_gpu, alpha=0.9)
ax.bar_label(b1, fmt='%.0f ms', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.0f ms', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=12)
ax.set_ylabel('Wall Time (ms)', fontsize=12)
ax.set_title('CPU vs GPU Wall Time by Option Type  (1M paths, 252 steps)', fontsize=12)
ax.legend(fontsize=11)
ax.set_yscale('log')
ax.grid(axis='y', which='both', alpha=0.3)
fig.tight_layout()
plt.show()

## Section 3 — Performance Summary Table

In [ ]:
hdr = (f"{'Option Type':<12} | {'CPU (ms)':>9} | {'GPU (ms)':>9} | "
       f"{'Speedup':>8} | {'GPU Mpaths/s':>12}")
sep = '-' * len(hdr)
print(sep)
print(hdr)
print(sep)
for name, t_cpu, t_gpu, spd, thr in results:
    print(f"{name:<12} | {t_cpu:>9.1f} | {t_gpu:>9.1f} | "
          f"{spd:>7.1f}× | {thr/1e6:>11.2f} M")
print(sep)